# 📊 NB2 — Análisis por Segmento
**Scotiabank Perú · Prevención de Fraude**  
Función reutilizable: `analizar_segmento(df, segmento)`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False})

# ── AJUSTAR ESTOS NOMBRES ──────────────────────────────────────
COL_MONTO    = 'MONTO'
COL_IND      = 'INDICADOR'
COL_SEGMENTO = 'SEGMENTO'
COL_SALDO    = 'ACFSALDO_DISPONIBLE_EN_MONEDA_TRX'   # <-- reemplaza con nombre real
COL_FECHA    = 'FECHA_HORA'
# ──────────────────────────────────────────────────────────────

COLORES = {'F':'#E63946','N':'#457B9D','G':'#2A9D8F','D':'#E9C46A','P':'#A8DADC'}
ORDEN_DIAS = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
NOMBRES_DIAS = {'Monday':'Lun','Tuesday':'Mar','Wednesday':'Mié',
                'Thursday':'Jue','Friday':'Vie','Saturday':'Sáb','Sunday':'Dom'}

def fmt_soles(ax, eje='x'):
    f = mticker.FuncFormatter(lambda x,_: f'S/ {x:,.0f}')
    if eje=='x': ax.xaxis.set_major_formatter(f)
    else: ax.yaxis.set_major_formatter(f)

print('✅ Config lista')

In [ ]:
df = pd.read_parquet('transferencias_consolidado.parquet')
df[COL_IND]      = df[COL_IND].astype(str).str.strip().str.upper()
df[COL_SEGMENTO] = df[COL_SEGMENTO].astype(str).str.strip()

# Saldo disponible → numérico
if COL_SALDO in df.columns:
    df[COL_SALDO] = (pd.to_numeric(
        df[COL_SALDO].astype(str).str.replace(',','',regex=False),
        errors='coerce'
    ))
    df['RATIO_MONTO_SALDO']   = df[COL_MONTO] / (df[COL_SALDO] + 1e-9)
    df['FLAG_VACIADO']        = (df['RATIO_MONTO_SALDO'] >= 0.80).astype(int)
    TIENE_SALDO = True
else:
    TIENE_SALDO = False
    print(f'⚠️  Columna saldo no encontrada: {COL_SALDO}')

print(f'Dataset cargado: {len(df):,} filas')
print(f'Segmentos disponibles: {sorted(df[COL_SEGMENTO].unique())}')

In [ ]:
# ══════════════════════════════════════════════════════════════
#  FUNCIÓN PRINCIPAL
# ══════════════════════════════════════════════════════════════

def analizar_segmento(df: pd.DataFrame, segmento: str) -> dict:
    """
    Análisis completo de fraude para un segmento dado.
    Capas: perfil → rango monto → temporal → saldo → resumen regla
    """

    sub = df[
        (df[COL_SEGMENTO] == str(segmento)) &
        (df[COL_IND].isin(['F','N','G']))
    ].copy()

    if len(sub) == 0:
        print(f'⚠️  Segmento {segmento} no encontrado o sin registros.')
        return {}

    fraudes = sub[sub[COL_IND]=='F']
    n_fraude = len(fraudes)
    n_total  = len(sub)
    fr       = n_fraude / n_total * 100

    print('\n' + '═'*60)
    print(f'  SEGMENTO {segmento}')
    print('═'*60)
    print(f'  Total transacciones : {n_total:,}')
    print(f'  Fraudes (F)         : {n_fraude:,}')
    print(f'  Fraud Rate          : {fr:.3f}%')
    if n_fraude > 0:
        print(f'  Mediana monto F     : S/ {fraudes[COL_MONTO].median():,.0f}')
        print(f'  Severidad (media)   : S/ {fraudes[COL_MONTO].mean():,.0f}')
        print(f'  Monto total fraude  : S/ {fraudes[COL_MONTO].sum():,.0f}')

    fig = plt.figure(figsize=(16, 20))
    fig.suptitle(f'Análisis de Fraude — Segmento {segmento}  '
                 f'(FR={fr:.3f}%  |  N fraudes={n_fraude})',
                 fontsize=14, fontweight='bold')

    # ── BLOQUE 1: Rango de monto ───────────────────────────────
    ax1 = fig.add_subplot(4, 2, 1)
    ax2 = fig.add_subplot(4, 2, 2)

    BINS   = [2000, 5000, 10000, 20000, 50000, np.inf]
    LABELS = ['2k–5k','5k–10k','10k–20k','20k–50k','50k+']
    sub['RANGO'] = pd.cut(sub[COL_MONTO], bins=BINS, labels=LABELS, right=False)

    rango_counts = sub.groupby(['RANGO', COL_IND]).size().unstack(fill_value=0)
    orden_ind = [i for i in ['F','G','N'] if i in rango_counts.columns]
    rango_counts[orden_ind].plot(
        kind='bar', stacked=True, ax=ax1,
        color=[COLORES.get(i,'#999') for i in orden_ind],
        edgecolor='white'
    )
    ax1.set_title('Volumen por Rango de Monto', fontweight='bold')
    ax1.set_xlabel('Rango (S/)'); ax1.set_ylabel('N')
    ax1.tick_params(axis='x', rotation=30)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:,.0f}'))

    # % fraude por rango
    rango_pct = rango_counts.div(rango_counts.sum(axis=1), axis=0) * 100
    if 'F' in rango_pct.columns:
        colores_rango = plt.cm.YlOrRd(
            rango_pct['F'] / max(rango_pct['F'].max(), 0.01)
        )
        ax2.bar(rango_pct.index, rango_pct['F'],
                color=colores_rango, edgecolor='white')
        for i, (rango, val) in enumerate(rango_pct['F'].items()):
            n_f = rango_counts.get('F', pd.Series()).get(rango, 0)
            ax2.text(i, val + 0.005, f'{val:.2f}%\n(N={n_f})',
                     ha='center', fontsize=8)
        ax2.set_title('Fraud Rate por Rango de Monto', fontweight='bold')
        ax2.set_xlabel('Rango (S/)'); ax2.set_ylabel('Fraud Rate (%)')
        ax2.tick_params(axis='x', rotation=30)

    # ── BLOQUE 2: Distribución visual monto F vs N ─────────────
    ax3 = fig.add_subplot(4, 2, 3)
    ax4 = fig.add_subplot(4, 2, 4)

    for ind in ['F','N','G']:
        s = sub.loc[sub[COL_IND]==ind, COL_MONTO]
        if len(s) < 5: continue
        sns.kdeplot(s, ax=ax3, label=ind, color=COLORES.get(ind,'#999'),
                    fill=True, alpha=0.2, linewidth=1.8)
    ax3.set_title('KDE Monto por Indicador', fontweight='bold')
    ax3.set_xlabel('Monto (S/)'); fmt_soles(ax3, 'x'); ax3.legend()

    bp_data  = [sub.loc[sub[COL_IND]==i, COL_MONTO].values
                for i in orden_ind if len(sub[sub[COL_IND]==i]) > 0]
    bp_labels = [i for i in orden_ind if len(sub[sub[COL_IND]==i]) > 0]
    if bp_data:
        bp = ax4.boxplot(bp_data, labels=bp_labels, patch_artist=True,
                         flierprops=dict(marker='o', markersize=2, alpha=0.3))
        for patch, ind in zip(bp['boxes'], bp_labels):
            patch.set_facecolor(COLORES.get(ind,'#999')); patch.set_alpha(0.7)
        ax4.set_title('Boxplot Monto por Indicador', fontweight='bold')
        ax4.set_ylabel('Monto (S/)'); fmt_soles(ax4, 'y')

    # ── BLOQUE 3: Temporal — hora ──────────────────────────────
    ax5 = fig.add_subplot(4, 2, 5)
    ax6 = fig.add_subplot(4, 2, 6)

    hora_stats = (sub
        .groupby('HORA')
        .agg(N_total=(COL_IND,'count'),
             N_fraude=(COL_IND, lambda x: (x=='F').sum()))
        .assign(FR=lambda x: x['N_fraude']/x['N_total']*100)
        .reset_index()
    )
    ax5.bar(hora_stats['HORA'], hora_stats['N_total'],
            color='#ADB5BD', label='Total')
    ax5.bar(hora_stats['HORA'], hora_stats['N_fraude'],
            color='#E63946', label='Fraude')
    ax5.set_title('Volumen por Hora', fontweight='bold')
    ax5.set_xlabel('Hora'); ax5.set_ylabel('N'); ax5.legend()
    ax5.set_xticks(range(0,24))

    media_fr_hora = hora_stats['FR'].mean()
    colores_hora = ['#E63946' if fr > media_fr_hora*2 else '#457B9D'
                    for fr in hora_stats['FR']]
    ax6.bar(hora_stats['HORA'], hora_stats['FR'],
            color=colores_hora, edgecolor='white')
    ax6.axhline(media_fr_hora, color='#E63946', linestyle='--', lw=1.5,
                label=f'Media: {media_fr_hora:.2f}%')
    ax6.set_title('Fraud Rate por Hora', fontweight='bold')
    ax6.set_xlabel('Hora'); ax6.set_ylabel('FR (%)'); ax6.legend()
    ax6.set_xticks(range(0,24))

    # ── BLOQUE 4: Heatmap hora × día ──────────────────────────
    ax7 = fig.add_subplot(4, 1, 3)

    pivot_hm = (sub
        .assign(FRAUDE_BIN=lambda x: (x[COL_IND]=='F').astype(int))
        .groupby(['DIA_SEMANA','HORA'])['FRAUDE_BIN']
        .mean() * 100
    ).unstack('HORA')
    pivot_hm = pivot_hm.reindex(
        [d for d in ORDEN_DIAS if d in pivot_hm.index]
    )
    pivot_hm.index = [NOMBRES_DIAS.get(d,d) for d in pivot_hm.index]

    sns.heatmap(pivot_hm, cmap='YlOrRd', annot=True, fmt='.1f',
                linewidths=0.3, ax=ax7, cbar_kws={'label':'FR%'},
                annot_kws={'size': 7})
    ax7.set_title('Fraud Rate (%) — Hora × Día de Semana', fontweight='bold')
    ax7.set_xlabel('Hora'); ax7.set_ylabel('')

    # ── BLOQUE 5: Saldo disponible ─────────────────────────────
    ax8 = fig.add_subplot(4, 2, 7)
    ax9 = fig.add_subplot(4, 2, 8)

    if TIENE_SALDO:
        for ind in ['F','N']:
            s = sub.loc[sub[COL_IND]==ind, 'RATIO_MONTO_SALDO'].dropna()
            s = s[s <= 2]   # recortar outliers extremos para visualizar
            if len(s) < 3: continue
            sns.kdeplot(s, ax=ax8, label=ind,
                        color=COLORES.get(ind,'#999'),
                        fill=True, alpha=0.2, linewidth=1.8)
        ax8.axvline(0.80, color='#E63946', linestyle='--', lw=1.5,
                    label='Umbral vaciado (80%)')
        ax8.set_title('KDE Ratio Monto/Saldo', fontweight='bold')
        ax8.set_xlabel('Ratio (1 = saldo completo transferido)')
        ax8.legend(fontsize=8)

        vaciado = (sub.groupby(COL_IND)['FLAG_VACIADO']
                     .mean() * 100
                     .reset_index()
                     .rename(columns={'FLAG_VACIADO':'Pct_Vaciado'}))
        ax9.bar(vaciado[COL_IND], vaciado['Pct_Vaciado'],
                color=[COLORES.get(i,'#999') for i in vaciado[COL_IND]],
                edgecolor='white')
        for _, row in vaciado.iterrows():
            ax9.text(row.name, row['Pct_Vaciado']+0.3,
                     f"{row['Pct_Vaciado']:.1f}%", ha='center', fontsize=9)
        ax9.set_title('% transacciones con vaciado ≥80%', fontweight='bold')
        ax9.set_ylabel('% del grupo'); ax9.set_xlabel('Indicador')
    else:
        ax8.text(0.5, 0.5, 'Saldo no disponible',
                 ha='center', va='center', transform=ax8.transAxes)
        ax9.set_visible(False)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fname = f'seg_{segmento}_analisis.png'
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {fname}')

    # ── Resumen de regla candidata ─────────────────────────────
    print(f'\n  📋 SEÑALES PARA REGLA — Segmento {segmento}')
    print('  ' + '-'*45)

    if n_fraude > 0:
        hora_top = hora_stats.loc[hora_stats['FR'].idxmax()]
        rango_top = rango_pct['F'].idxmax() if 'F' in rango_pct.columns else 'N/D'
        print(f'  Hora con mayor FR      : {int(hora_top["HORA"]):02d}:00h '
              f'({hora_top["FR"]:.2f}%)')
        print(f'  Rango de monto top     : {rango_top}')

        if TIENE_SALDO:
            pct_vaciado_F = fraudes['FLAG_VACIADO'].mean() * 100
            pct_vaciado_N = sub.loc[sub[COL_IND]=='N','FLAG_VACIADO'].mean()*100
            print(f'  % vaciado en FRAUDE    : {pct_vaciado_F:.1f}%')
            print(f'  % vaciado en NORMAL    : {pct_vaciado_N:.1f}%')
            sep_vaciado = pct_vaciado_F - pct_vaciado_N
            if sep_vaciado > 10:
                print(f'  → FLAG_VACIADO discrimina (+{sep_vaciado:.1f}pp vs normal) ✅')
            else:
                print(f'  → FLAG_VACIADO no discrimina ({sep_vaciado:.1f}pp) ❌')

    print()
    return {'sub': sub, 'fraudes': fraudes, 'hora_stats': hora_stats}


print('✅ Función analizar_segmento() lista')

## Uso — ejecutar por segmento

In [ ]:
# Segmento 31 — el de mayor volumen de fraudes
res_31 = analizar_segmento(df, '31')

In [ ]:
# Segmento 33
res_33 = analizar_segmento(df, '33')

In [ ]:
# Segmento 32
res_32 = analizar_segmento(df, '32')

In [ ]:
# Segmento 34
res_34 = analizar_segmento(df, '34')

## Comparativa final entre segmentos

In [ ]:
# Tabla resumen comparativa de los 4 segmentos
SEGS = ['31','32','33','34']
filas = []
for seg in SEGS:
    s = df[(df[COL_SEGMENTO]==seg) & (df[COL_IND].isin(['F','N','G']))]
    f = s[s[COL_IND]=='F']
    if len(s) == 0: continue
    fila = {
        'Segmento'         : seg,
        'N Total'          : f'{len(s):,}',
        'N Fraudes'        : len(f),
        'Fraud Rate'       : f'{len(f)/len(s)*100:.3f}%',
        'Monto Total F'    : f'S/ {f[COL_MONTO].sum():,.0f}',
        'Severidad'        : f'S/ {f[COL_MONTO].mean():,.0f}' if len(f)>0 else '-',
        'Mediana F'        : f'S/ {f[COL_MONTO].median():,.0f}' if len(f)>0 else '-',
    }
    if TIENE_SALDO and len(f)>0:
        fila['% Vaciado F'] = f"{f['FLAG_VACIADO'].mean()*100:.1f}%"
    filas.append(fila)

comparativa = pd.DataFrame(filas).set_index('Segmento')
print('\n📊 COMPARATIVA DE SEGMENTOS')
display(comparativa)